In [7]:
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from itertools import combinations

df = pd.read_csv("/content/campus_store_transactions - campus_store_transactions.csv")
df['L'] = df['Items'].apply(lambda x: x.split('|'))

te = TransactionEncoder()
ohe = pd.DataFrame(te.fit(df['L']).transform(df['L']), columns=te.columns_)

txM = df[df['TimeOfDay'] == 'Morning']['L'].tolist()
txA = df[df['TimeOfDay'] == 'Afternoon']['L'].tolist()
txE = df[df['TimeOfDay'] == 'Evening']['L'].tolist()
txN = df[df['TimeOfDay'] == 'Night']['L'].tolist()

In [8]:
class Node:
    def __init__(self, i, c, p):
        self.i, self.c, self.p, self.ch, self.lnk = i, c, p, {}, None

def b_tree(db, min_c):
    hd = {}
    for t, c in db.items():
        for i in t: hd[i] = hd.get(i, 0) + c
    hd = {k: [v, None] for k, v in hd.items() if v >= min_c}
    f_itms = set(hd.keys())
    if not f_itms: return None, None

    rt = Node('R', 1, None)
    for t, c in db.items():
        cur_i = {x: hd[x][0] for x in t if x in f_itms}
        if cur_i:
            s_itms = [v[0] for v in sorted(cur_i.items(), key=lambda x: (-x[1], x[0]))]
            cur_n = rt
            for i in s_itms:
                if i in cur_n.ch:
                    cur_n.ch[i].c += c
                else:
                    n_node = Node(i, c, cur_n)
                    cur_n.ch[i] = n_node
                    if hd[i][1] is None:
                        hd[i][1] = n_node
                    else:
                        tmp = hd[i][1]
                        while tmp.lnk: tmp = tmp.lnk
                        tmp.lnk = n_node
                cur_n = cur_n.ch[i]
    return rt, hd

def m_tree(hd, min_c, pfx, out):
    s_itms = [v[0] for v in sorted(hd.items(), key=lambda x: x[1][0])]
    for i in s_itms:
        n_pfx = pfx | {i}
        out[frozenset(n_pfx)] = hd[i][0]

        cpb = {}
        n = hd[i][1]
        while n:
            path, tmp = [], n.p
            while tmp and tmp.i != 'R':
                path.append(tmp.i)
                tmp = tmp.p
            if path: cpb[frozenset(path)] = n.c
            n = n.lnk

        c_rt, c_hd = b_tree(cpb, min_c)
        if c_rt: m_tree(c_hd, min_c, n_pfx, out)

res = {}
for seg, txs in [('M', txM), ('A', txA), ('E', txE), ('N', txN)]:
    n_tx = len(txs)
    mc = 0.2 * n_tx
    db = {}
    for t in txs: db[frozenset(t)] = db.get(frozenset(t), 0) + 1
    rt, hd = b_tree(db, mc)
    f_sets = {}
    if rt: m_tree(hd, mc, set(), f_sets)
    res[seg] = {k: v / n_tx for k, v in f_sets.items()}

In [9]:
for seg in ['M', 'A', 'E', 'N']:
    f_sets = res[seg]
    rules = []
    for i_set, sup in f_sets.items():
        if len(i_set) > 1:
            for i in range(1, len(i_set)):
                for a in combinations(i_set, i):
                    ant = frozenset(a)
                    con = i_set - ant
                    if ant in f_sets and con in f_sets:
                        cf = sup / f_sets[ant]
                        lf = cf / f_sets[con]
                        if cf >= 0.6 and lf >= 1.0:
                            rules.append((ant, con, cf, lf))

    print(f"[{seg}] FreqSets: {len(f_sets)} | Rules: {len(rules)}")
    for k, v in list(f_sets.items())[:2]:
        print(f"  Set: {set(k)} (s:{v:.2f})")
    for r in rules[:2]:
        print(f"  Rule: {set(r[0])} -> {set(r[1])} (c:{r[2]:.2f}, l:{r[3]:.2f})")

[M] FreqSets: 11 | Rules: 10
  Set: {'Stapler'} (s:0.20)
  Set: {'Pen', 'Stapler'} (s:0.20)
  Rule: {'Stapler'} -> {'Pen'} (c:1.00, l:1.25)
  Rule: {'Stapler'} -> {'Pen', 'Notebook'} (c:1.00, l:1.25)
[A] FreqSets: 17 | Rules: 21
  Set: {'Pen'} (s:0.20)
  Set: {'Pen', 'Notebook'} (s:0.20)
  Rule: {'Pen'} -> {'Notebook'} (c:1.00, l:5.00)
  Rule: {'Notebook'} -> {'Pen'} (c:1.00, l:5.00)
[E] FreqSets: 16 | Rules: 11
  Set: {'Juice'} (s:0.20)
  Set: {'Cookies', 'Juice'} (s:0.20)
  Rule: {'Juice'} -> {'Cookies'} (c:1.00, l:1.25)
  Rule: {'Juice'} -> {'Cookies', 'Coffee'} (c:1.00, l:1.67)
[N] FreqSets: 7 | Rules: 5
  Set: {'Chips'} (s:0.60)
  Set: {'Chips', 'Energy Drink'} (s:0.60)
  Rule: {'Chips'} -> {'Energy Drink'} (c:1.00, l:1.00)
  Rule: {'Energy Drink'} -> {'Chips'} (c:0.60, l:1.00)


### Conclusion: Process & Results Overview
In this assignment, we successfully implemented the FP-Growth algorithm and association rule mining entirely from scratch to analyze temporal customer purchasing behavior. By segmenting the dataset by time of day, manually constructing FP-Trees, and recursively mining conditional pattern bases (minimum support = 0.2, minimum confidence = 0.6), we uncovered distinct shifts in student habits. The process reveals that basket complexity peaks in the afternoon (17 frequent sets, 21 rules) with highly correlated cross-category items yielding strong lift values (e.g., Pens $\rightarrow$ Notebooks, lift = 5.00). Conversely, morning and night segments are highly predictable and utilitarian, anchored by perfect-confidence rules for stationary in the morning (Stapler $\rightarrow$ Pen) and late-night study snacks at night (Chips $\rightarrow$ Energy Drink), demonstrating the efficiency of FP-Growth in extracting behavioral contexts without relying on external libraries.